# 🔗 03 · 链路预测与图对比学习

> 推荐/社交/风控里最常用的图任务是**链路预测**（预测两个节点是否会有边/交互）。
> 本节手写**基于结构相似度的链路预测**（Common Neighbors/Adamic-Adar）与
> **图对比学习（GraphCL 思想：扰动+InfoNCE）**。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 链路预测是什么？ | 预测未观测边是否存在 |
| 2 | 结构特征怎么算？ | 共同邻居/AA/Jaccard |
| 3 | 图对比学习？ | 图扰动 + 对比损失 |
| 4 | 应用场景？ | 推荐/好友推荐/异常检测 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
import networkx as nx
import sqlite3
import os
rng = np.random.default_rng(42)


## 1️⃣ 链路预测任务设定

In [2]:
G = nx.karate_club_graph()
nodes = list(G.nodes())
n = len(nodes)
print('节点:', G.number_of_nodes(), '现有边:', G.number_of_edges())
# 候选：所有未连接的节点对（预测哪些未来会相连）
non_edges = list(nx.non_edges(G))
print('未连接节点对:', len(non_edges))
print('➡️ 目标：给这些未连接对打分，分数高=未来更可能连边')

def node_emb_simple(G, nodes, dim=8):
    # 简化表示：节点特征=度数+邻居度均值（够区分即可）；用固定 seed 保证可复现
    r = np.random.default_rng(7)
    emb = np.zeros((len(nodes), dim))
    for i, nd in enumerate(nodes):
        deg = G.degree(nd)
        nb_deg = np.mean([G.degree(w) for w in G.neighbors(nd)]) if G.degree(nd) > 0 else 0
        emb[i] = [deg / 10, nb_deg / 10] + list(r.normal(0, 0.05, dim - 2))
    return emb

节点: 34 现有边: 78
未连接节点对: 483
➡️ 目标：给这些未连接对打分，分数高=未来更可能连边


## 2️⃣ 结构相似度指标（手写）

In [3]:
def common_neighbors(G, u, v):
    return len(set(G.neighbors(u)) & set(G.neighbors(v)))

def adamic_adar(G, u, v):
    common = set(G.neighbors(u)) & set(G.neighbors(v))
    return sum(1.0 / np.log(len(list(G.neighbors(w))) + 1e-9) for w in common)

# 取前 10 个未连接对打分
pairs = non_edges[:10]
for u, v in pairs:
    cn = common_neighbors(G, u, v)
    aa = adamic_adar(G, u, v)
    print(f'({u},{v}) 共同邻居={cn:>2} Adamic-Adar={aa:.3f}')
print('\n➡️ 共同邻居多 -> 更可能连边（社交"共同好友"直觉）')

(0,32) 共同邻居= 3 Adamic-Adar=1.614
(0,33) 共同邻居= 4 Adamic-Adar=2.711
(0,9) 共同邻居= 1 Adamic-Adar=0.434
(0,14) 共同邻居= 0 Adamic-Adar=0.000
(0,15) 共同邻居= 0 Adamic-Adar=0.000
(0,16) 共同邻居= 2 Adamic-Adar=1.443
(0,18) 共同邻居= 0 Adamic-Adar=0.000
(0,20) 共同邻居= 0 Adamic-Adar=0.000
(0,22) 共同邻居= 0 Adamic-Adar=0.000
(0,23) 共同邻居= 0 Adamic-Adar=0.000

➡️ 共同邻居多 -> 更可能连边（社交"共同好友"直觉）


## 3️⃣ 用已知边评估指标有效性

In [4]:
# 留出法：隐藏 20% 边当正样本，随机负样本，看指标能否区分
all_edges = list(G.edges())
rng.shuffle(all_edges)
k = int(len(all_edges) * 0.2)
pos_test = all_edges[:k]
train_G = G.copy()
train_G.remove_edges_from(pos_test)

pos_scores = [adamic_adar(train_G, u, v) for u, v in pos_test]
neg_pairs = rng.choice(len(list(nx.non_edges(train_G))), size=k, replace=False)
neg_list = list(nx.non_edges(train_G))
neg_scores = [adamic_adar(train_G, u, v) for i, (u, v) in enumerate(neg_list) if i in neg_pairs]

from sklearn.metrics import roc_auc_score
y = np.array([1] * k + [0] * k)
scores = np.array(pos_scores + neg_scores)
auc = roc_auc_score(y, scores)
print(f'链路预测 AUC（Adamic-Adar）: {auc:.3f}')
assert auc > 0.6
print('✅ 结构相似度指标能有效预测连边（>0.5 随机）')

链路预测 AUC（Adamic-Adar）: 0.738
✅ 结构相似度指标能有效预测连边（>0.5 随机）


## 4️⃣ 图对比学习：扰动 + InfoNCE（简化）

In [5]:
# GraphCL 思想：同一张图随机扰动（删边/加噪特征）成两个视图，拉近视图间节点
def corrupt_graph(G, drop_frac=0.1, seed=42):
    r = np.random.default_rng(seed)
    Gc = G.copy()
    edges = list(Gc.edges())
    drop = r.choice(len(edges), size=int(len(edges) * drop_frac), replace=False)
    Gc.remove_edges_from([edges[i] for i in drop])
    return Gc

emb1 = node_emb_simple(G, nodes)
emb2 = node_emb_simple(corrupt_graph(G), nodes)
# InfoNCE：自身视图对为"正"，与其他节点为"负"
sim = (emb1 / np.linalg.norm(emb1, axis=1, keepdims=True)) @       (emb2 / np.linalg.norm(emb2, axis=1, keepdims=True)).T
diag_rank = (sim.argmax(axis=1) == np.arange(n)).mean()
print(f'对比视图"节点自匹配"比例: {diag_rank:.1%}')
assert diag_rank > 0.8
print('✅ 图对比学习：扰动后同一节点仍能互相匹配（表示稳定）')

对比视图"节点自匹配"比例: 88.2%
✅ 图对比学习：扰动后同一节点仍能互相匹配（表示稳定）


## 5️⃣ 链路预测在工业中的应用

In [6]:
print('''1) 推荐系统：用户-物品二部图连边预测 -> 召回候选；
2) 社交：好友推荐（共同好友）、关注预测；
3) 风控：异常账户-设备连边预测、团伙识别；
4) 知识图谱：关系补全（缺失三元组预测）-> 见 05-知识图谱推理；
5) 药研：蛋白-分子交互预测。
''')

1) 推荐系统：用户-物品二部图连边预测 -> 召回候选；
2) 社交：好友推荐（共同好友）、关注预测；
3) 风控：异常账户-设备连边预测、团伙识别；
4) 知识图谱：关系补全（缺失三元组预测）-> 见 05-知识图谱推理；
5) 药研：蛋白-分子交互预测。



## 📝 自测清单

- [ ] 能手写共同邻居 / Adamic-Adar 指标
- [ ] 能用留出法评估链路预测（AUC）
- [ ] 能解释图对比学习（扰动+InfoNCE）思想
- [ ] 能举 4 个链路预测应用场景